In [1]:
!pip install pyedflib pandas scipy


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 32.1 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for pyedflib: filename=pyedflib-0.1.40-cp311-cp311-linux_x86_64.whl size=2746680 sha256=023a113c3545f8cf4c767e173d0487bbae3c09470527294b4ab2259d86fa6883
  Stored in directory: /root/.cache/pip/wheels/8d/df/d6/88ce619bde055ebffebae5380645802eca490817853b60b45b
Successfully built pyedflib


In [2]:
import os
import numpy as np
import pandas as pd
import scipy.io as sio
import pyedflib
from google.colab import drive


In [ ]:
drive.mount('/content/drive')

In [ ]:
input_folder = '/content/drive/MyDrive/TEST'  # Replace with your folder
output_folder = '/content/drive/MyDrive/ProcessedFiles'
os.makedirs(output_folder, exist_ok=True)

In [ ]:
def time_to_sec(t):
    hh, mm, ss = [int(x) for x in t.split('.')]
    if hh < 12:
        hh += 24
    return hh * 3600 + mm * 60 + ss


In [ ]:
files = os.listdir(input_folder)
edf_files = sorted([f for f in files if f.endswith('.edf')])
txt_files = sorted([f for f in files if f.endswith('.txt')])
common_files = set(f[:-4] for f in edf_files) & set(f[:-4] for f in txt_files)

In [ ]:
sample_rate = 512
min_duration = 2 * sample_rate


In [ ]:
base= "brux1"
print(f"\nProcessing {base}...")

edf_path = os.path.join(input_folder, base + '.edf')
txt_path = os.path.join(input_folder, base + '.txt')

# Find the line where the table starts
with open(txt_path, 'r') as file:
    lines = file.readlines()

# Find the header line
for i, line in enumerate(lines):
    if line.strip().startswith("Sleep Stage") or "Time" in line and "Event" in line:
        data_start_idx = i + 1  # Start after header
        break
else:
    raise ValueError(f"Data table not found in {txt_path}")

# Read the actual table part only
from io import StringIO
data_str = ''.join(lines[data_start_idx:])
df = pd.read_csv(StringIO(data_str), delimiter='\t',
                names=['SleepStage', 'Position', 'Time', 'Event', 'Duration', 'Location'],
                header=None)

df['Duration'] = pd.to_numeric(df['Duration'], errors='coerce').fillna(0)
df = df.dropna(subset=['Time', 'Event'])




In [ ]:
print(df.head(10))

In [ ]:
!pip install mne # For MNE-Python
!pip install neo # For Neo

In [ ]:
import mne

try:
    raw = mne.io.read_raw_edf(edf_path, preload=True, stim_channel=None, verbose=True)

    labels = raw.ch_names

    start_time = raw.info['meas_date'].time()

    start_sec = start_time.hour * 3600 + start_time.minute * 60 + start_time.second

    print("File read successfully with MNE-Python.")
    print(f"Signal Labels: {labels}")
    print(f"Start Time: {start_time}")
    print(f"Start Time in Seconds: {start_sec}")

except FileNotFoundError:
    print(f"Error: File not found at {edf_path}")
except Exception as e:
    print(f"An error occurred while reading the file with MNE-Python: {e}")

Extracting EDF parameters from /content/drive/MyDrive/TEST/brux1.edf...
EDF file detected
Setting channel info structure...
Creating raw.info structure...


<ipython-input-11-2733613849>:4: RuntimeWarning: Number of records from the header does not match the file size (perhaps the recording was not stopped before exiting). Inferring from the file size.
  raw = mne.io.read_raw_edf(edf_path, preload=True, stim_channel=None, verbose=True)
<ipython-input-11-2733613849>:4: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=True, stim_channel=None, verbose=True)
<ipython-input-11-2733613849>:4: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_edf(edf_path, preload=True, stim_channel=None, verbose=True)


Reading 0 ... 7342591  =      0.000 ... 14340.998 secs...
File read successfully with MNE-Python.
Signal Labels: ['Fp2-F4', 'F4-C4', 'C4-P4', 'P4-O2', 'F8-T4', 'T4-T6', 'FP1-F3', 'F3-C3', 'C3-P3', 'P3-O1', 'F7-T3', 'T3-T5', 'C4-A1', 'ROC-LOC', 'EMG1-EMG2', 'ECG1-ECG2', 'DX1-DX2', 'SX1-SX2']
Start Time: 21:54:57
Start Time in Seconds: 78897


In [ ]:
try:
    idx_c = labels.index('F4-C4')
except ValueError:
    print(f"F4-C4 channel not found in {base}, skipping...")

In [ ]:
idx_c = raw.ch_names.index('F4-C4')

val = raw.get_data(picks=[idx_c])[0]

print(f"Successfully read signal for channel index {idx_c}.")
print(len(val))
print(raw.info['sfreq'])

Successfully read signal for channel index 1.
7342592
512.0


In [ ]:
time_sec = df['Time'].apply(lambda t: time_to_sec(str(t)))
df['time_sec_loc'] = time_sec - start_sec
print(df['time_sec_loc'][0])

1400


In [ ]:
def extract_chunks(event_type, label_num):
    data_out = []
    matches = df[df['Event'].str.contains(event_type)]
    print(matches.head(10))
    print(len(matches))
    k=0
    for _, row in matches.iterrows():
        k+=1
        start = int(row['time_sec_loc'] * sample_rate)
        end = start + int(row['Duration'] * sample_rate)

        segment = val[start:end]
        if(len(segment) ==0):
            print(row['time_sec_loc'])
            print(k)
            print(len(segment))
            print(row['Duration'])
            print(start," ",end)
        if len(segment) >= min_duration:
            segment = segment[:(len(segment)//min_duration)*min_duration]
            reshaped = np.reshape(segment, (-1, min_duration))
            labeled = np.hstack([reshaped, np.full((reshaped.shape[0], 1), label_num)])
            data_out.append(labeled)
    if data_out:
        return np.vstack(data_out)
    else:
        return np.empty((0, min_duration + 1))


In [ ]:
data_A1 = extract_chunks('MCAP-A1', 1)


    SleepStage          Position      Time    Event  Duration    Location  \
36          S2  Unknown Position  22.34.08  MCAP-A1         4  EEG-Fp2-F4   
47          S2  Unknown Position  22.37.41  MCAP-A1         3  EEG-Fp2-F4   
91          S2  Unknown Position  22.52.14  MCAP-A1         6  EEG-Fp2-F4   
96          S2  Unknown Position  22.53.41  MCAP-A1         3  EEG-Fp2-F4   
98          S2  Unknown Position  22.53.59  MCAP-A1        11  EEG-Fp2-F4   
100         S2  Unknown Position  22.54.42  MCAP-A1         6  EEG-Fp2-F4   
103         S3  Unknown Position  22.55.34  MCAP-A1         6  EEG-Fp2-F4   
105         S3  Unknown Position  22.55.53  MCAP-A1         5  EEG-Fp2-F4   
107         S3  Unknown Position  22.56.29  MCAP-A1         5  EEG-Fp2-F4   
109         S3  Unknown Position  22.57.01  MCAP-A1         6  EEG-Fp2-F4   

     time_sec_loc  
36           2351  
47           2564  
91           3437  
96           3524  
98           3542  
100          3585  
103         

In [ ]:
print(len(data_A1))

403


In [ ]:
len(data_A1[0])

1025